In [2]:
# import packages

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import math
from scipy.misc import electrocardiogram
from scipy.signal import find_peaks
from scipy import optimize
from scipy.stats import gaussian_kde
from scipy.optimize import curve_fit
from scipy.interpolate import interpn
from scipy.interpolate import interp1d

import tkinter as tk
import tkinter.filedialog
import tkinter.messagebox
import os
import glob
import time

from ast import literal_eval


### Execute the next cell if you have a jupyter dark theme
Don't if you don't or don't know what this means

https://medium.com/datadriveninvestor/how-can-i-customize-jupyter-notebook-into-dark-mode-7985ce780f38

In [5]:
from jupyterthemes import jtplot
jtplot.style(theme='onedork', context='notebook', ticks=True, grid=False)

#restart the core in case you executed the cell in white theme and cannot see ticks anymore

ModuleNotFoundError: No module named 'jupyterthemes'

In [7]:
  
def gaussian(x, height, center, width):
    return height*np.exp(-(x - center)**2/(2*width**2))

def gaussian_fit_simple_results(data, oversampling = 10, guessed_width = 20):
    
    #########
    #########fit gaussian in left window to find center of band
    #########
    min_x =  np.min(data)
    max_x =  np.max(data)
    if math.isnan(max_x-min_x) or round((max_x-min_x)*oversampling)==0:
        pars, hData_filtered, hData_filtered_x = [], [], []
    else:  
        bins_single = int(round((max_x-min_x)*oversampling))
        # histogram of positions along the x axis
        h_filtered = np.histogram(data)#, bins = bins_single)  
        #histogram height
        hData_filtered = h_filtered[0]
        ###get the center of each bin
        # bin size
        step = (h_filtered[1][1]-h_filtered[1][0])
        hData_filtered_x = h_filtered[1][:len(h_filtered[1])-1]+step/2
        #guess that band is centered on the window
        guess = [np.mean(hData_filtered), np.mean(hData_filtered_x), guessed_width]
        
        try:
            pars, cov = curve_fit(f=gaussian, xdata=hData_filtered_x, ydata=hData_filtered, p0=guess, bounds=(-np.inf, np.inf))
        except:
            pars = []
    return pars, hData_filtered, hData_filtered_x

## Load result of analysis

In [17]:
root = tk.Tk()
loc_filename = tk.filedialog.askopenfilename(parent=root, 
                                             title='Please select the file containing the result of your analysis')
root.withdraw()

all_results = pd.read_csv(loc_filename)

#create a new column to estimate sarcomere length later on
sarcomere_lengths = [0.0]*len(all_results)
all_results['sarcomere_length'] = sarcomere_lengths
all_results.head(3)

,day,year,sample,nanobody,fly_number,selection,probe,dye,zdisk,band_distance,band_positions,band_widths,band_widths_estimate,sarcomere_length
0,20240119,2024,1,slsGFP,2,1,P3,Cy3B,1,0,[16.81506414777755],0.218101,[0.19585868056749886],0.0
1,20240119,2024,1,slsGFP,2,1,P3,Cy3B,2,0,[50.73842453060701],0.246616,[0.28725380047925336],0.0
2,20240119,2024,1,slsGFP,2,1,P3,Cy3B,3,0,[84.58001148871212],0.283354,[0],0.0


## Estimate of sarcomere length

In [20]:
pixel_size = 97

samples = all_results['sample'].unique()
FOVs = all_results.fly_number.unique()
nanobodies = all_results.nanobody.unique()
selections = all_results.selection.unique()
days = all_results.day.unique()
  
for n in days:
    cond5 = all_results['day'] == n
    for i in samples:
        cond1 = all_results['sample'] == i
        for j in FOVs:
            cond2 = all_results['fly_number'] == j
            for k in nanobodies:
                cond3 = all_results['nanobody'] == k
                for l in selections:
                    cond4 = all_results['selection'] == l
                    selection = all_results[cond1 & cond2 & cond3 & cond4 & cond5]
                    positions = selection['band_positions'].apply(literal_eval).to_numpy()
                    zdisks = selection['zdisk']
                    if len(zdisks) > 1:
                        indices = all_results[cond1 & cond2 & cond3 & cond4 & cond5].index
                        sarcomere_length = []
                        for m in range(len(zdisks)-1):
                            #print(m)
                            left = (positions[m][0])
                            #print(left)
                            right = (positions[m+1][0])
                            #print(right)
                            sarcomere_length = pixel_size*(right-left)
                            #print(type(sarcomere_length))
                            all_results.at[indices[m+1], 'sarcomere_length'] = sarcomere_length
                
all_results.head()

,day,year,sample,nanobody,fly_number,selection,probe,dye,zdisk,band_distance,band_positions,band_widths,band_widths_estimate,sarcomere_length
0,20240119,2024,1,slsGFP,2,1,P3,Cy3B,1,0,[16.81506414777755],0.218101,[0.19585868056749886],0.000000
1,20240119,2024,1,slsGFP,2,1,P3,Cy3B,2,0,[50.73842453060701],0.246616,[0.28725380047925336],3290.565957
2,20240119,2024,1,slsGFP,2,1,P3,Cy3B,3,0,[84.58001148871212],0.283354,[0],3282.633935
3,20240119,2024,1,slsGFP,2,1,P3,Cy3B,4,0,[118.33593623734531],0.316868,[0.35205407937118294],3274.324701
4,20240119,2024,1,slsGFP,2,1,P3,Cy3B,5,0,[152.0385106024022],0.177846,[0],3269.149713


In [22]:
# Save results with sarcomere length to avoid extracting sarcomere length each time
filename_with_sarc = r'/Volumes/dataSchnorrer/EqpSchnorrer/Aditya/Experiments/SlsGFP/SlsGFP_P3_Cy3B_All_results_Combined_with_sarc_length_2026.csv'
all_results.to_csv(filename_with_sarc)